### Data Loader

Copies one genereated batch from the `dev.stripex.staging` volume into the `dev.stepright.landing` volume, and verifies row counts per source.

**Before running:** manually upload batch folder into `dev.stripex.staging/batch_<n>/` using Databricks Workspace UI 

In [0]:
dbutils.widgets.text("batch", "0", "Batch number to load")
batch_num = dbutils.widgets.get("batch")

staging_root = f"/Volumes/dev/stripex/staging/batch_{batch_num}"
landing_root = f"/Volumes/dev/stripex/landing"

print(f"Loading batch_{batch_num} from {staging_root} into {landing_root}")

In [0]:
subfolders = [
    "orders_cdc",
    "order_items_cdc",
    "customers_cdc",
    "products",
    "categories",
    "clickstream",
    "inventory",
]

for folder in subfolders:
    src = f"{staging_root}/{folder}"
    dst = f"{landing_root}/{folder}"
    try:
        dbutils.fs.cp(src, dst, recurse=True)
        print(f"Copied: {src} -> {dst}")
    except Exception as e:
        print(f"Skipped {folder} (nothing staged for this batch: {e})")

### Verify Row counts per source

In [0]:
csv_sources = ["products", "categories", "inventory"]
json_sources = ["orders_cdc", "order_items_cdc", "customers_cdc", "clickstream"]

print(f"{'source':<20}{'format':<8}{'row_count'}")
print("-" * 45)

for folder in csv_sources:
    path = f"{landing_root}/{folder}"
    try:
        count = spark.read.option("header", "true").csv(path).count()
    except Exception:
        count = 0
    print(f"{folder:<20}{'csv':<8}{count}")
 
for folder in json_sources:
    path = f"{landing_root}/{folder}"
    try:
        count = spark.read.json(path).count()
    except Exception:
        count = 0
    print(f"{folder:<20}{'json':<8}{count}")

**Now all source files are copied from `staging` volume into `landing` volume. If it returns 0 rows, check if copy process completed successfully into `landing` zone.**